In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
%load_ext autoreload
%autoreload 2

from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.ve_surface import BASE_MAP_SURFACE, TRUE_SURFACE, ve_table

train = generate_drive_cycle()
test = generate_drive_cycle(cycle=VALIDATION_CYCLE)
base = ve_table(params=BASE_MAP_SURFACE)
truth = ve_table(params=TRUE_SURFACE)

In [ ]:
from vetuner.iteration import iterate_calibration
from vetuner.metrics import evaluate_afr, evaluate_table
from vetuner.sensors import SensorConfig

test = generate_drive_cycle(cycle=VALIDATION_CYCLE)
truth = ve_table(params=TRUE_SURFACE)

def run(sensor_config=SensorConfig(), **kwargs):
    history = iterate_calibration(
        train, base, gain=0.6, max_passes=4, method="gaussian_process",
        sensor_config=sensor_config, **kwargs,
    )
    return (
        evaluate_table(history.final_table, truth).rmse,
        evaluate_afr(history.final_table, test).mean_abs_pct,
    )

baseline_rmse, baseline_afr = run()
print(f"baseline: RMSE {baseline_rmse:.2f}, AFR {baseline_afr:.2f}%\n")

print(f"{'parameter':28s} {'value':>8} {'RMSE':>7} {'AFR%':>7} {'dAFR':>7}")
for noise in (0.0, 0.05, 0.10, 0.20, 0.40):
    rmse, afr = run(sensor_config=SensorConfig(afr_noise_std=noise))
    print(f"{'AFR noise (units)':28s} {noise:8.2f} {rmse:7.2f} {afr:7.2f} {afr - baseline_afr:+7.2f}")

for tau in (0.0, 0.06, 0.12, 0.25, 0.50):
    rmse, afr = run(sensor_config=SensorConfig(afr_tau_s=tau))
    print(f"{'AFR sensor tau (s)':28s} {tau:8.2f} {rmse:7.2f} {afr:7.2f} {afr - baseline_afr:+7.2f}")

for bias in (0.0, 1.0, 2.0, 5.0):
    rmse, afr = run(sensor_config=SensorConfig(map_bias_kpa=bias))
    print(f"{'MAP bias (kPa)':28s} {bias:8.1f} {rmse:7.2f} {afr:7.2f} {afr - baseline_afr:+7.2f}")